In [3]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Aryan@2010",  # ← your password
    database="movie_recommender"
)
cursor = conn.cursor()

# Load from Bronze
movies  = pd.read_sql("SELECT * FROM bronze_movies", conn)
ratings = pd.read_sql("SELECT * FROM bronze_ratings", conn)
tags    = pd.read_sql("SELECT * FROM bronze_tags", conn)

print("✅ Loaded from Bronze!")
print(movies.shape, ratings.shape, tags.shape)

✅ Loaded from Bronze!
(9742, 3) (100836, 4) (3683, 4)


/var/folders/qk/tl_zqdz504d2h66y82f8tl740000gn/T/ipykernel_15375/760766142.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  movies  = pd.read_sql("SELECT * FROM bronze_movies", conn)
/var/folders/qk/tl_zqdz504d2h66y82f8tl740000gn/T/ipykernel_15375/760766142.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ratings = pd.read_sql("SELECT * FROM bronze_ratings", conn)
/var/folders/qk/tl_zqdz504d2h66y82f8tl740000gn/T/ipykernel_15375/760766142.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tags    = pd.read_sql("SEL

In [4]:
# Extract year from title
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')

# Clean title (remove year)
movies['clean_title'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()

# Split genres into list
movies['genres'] = movies['genres'].str.replace('|', ', ', regex=False)

# Drop movies with no genres
movies_clean = movies[movies['genres'] != '(no genres listed)'].copy()

print("✅ Movies cleaned!")
print(movies_clean.head(3))

✅ Movies cleaned!
   movieId                    title  \
0        1         Toy Story (1995)   
1        2           Jumanji (1995)   
2        3  Grumpier Old Men (1995)   

                                            genres  year       clean_title  
0  Adventure, Animation, Children, Comedy, Fantasy  1995         Toy Story  
1                     Adventure, Children, Fantasy  1995           Jumanji  
2                                  Comedy, Romance  1995  Grumpier Old Men  


In [5]:
import pandas as pd

# Convert timestamp to readable date
ratings['date'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings_clean = ratings.drop(columns=['timestamp'])

# Remove any extreme outliers
ratings_clean = ratings_clean[(ratings_clean['rating'] >= 0.5) & 
                               (ratings_clean['rating'] <= 5.0)]

print("✅ Ratings cleaned!")
print(ratings_clean.head(3))

✅ Ratings cleaned!
   userId  movieId  rating                date
0       1        1     4.0 2000-07-30 18:45:03
1       1        3     4.0 2000-07-30 18:20:47
2       1        6     4.0 2000-07-30 18:37:04


In [9]:
# Create Silver movies table
cursor.execute("DROP TABLE IF EXISTS silver_movies")
cursor.execute("""
    CREATE TABLE silver_movies (
        movieId INT,
        title VARCHAR(255),
        genres VARCHAR(500),
        clean_title VARCHAR(255),
        year INT
    )
""")

for _, row in movies_clean.iterrows():
    # Safely handle missing year
    try:
        year = int(row['year']) if pd.notna(row['year']) else None
    except:
        year = None

    cursor.execute(
        "INSERT INTO silver_movies VALUES (%s, %s, %s, %s, %s)",
        (int(row['movieId']), row['title'], row['genres'], 
         row['clean_title'], year)
    )

# Create Silver ratings table
cursor.execute("DROP TABLE IF EXISTS silver_ratings")
cursor.execute("""
    CREATE TABLE silver_ratings (
        userId INT,
        movieId INT,
        rating FLOAT,
        date DATETIME
    )
""")

for _, row in ratings_clean.iterrows():
    cursor.execute(
        "INSERT INTO silver_ratings VALUES (%s, %s, %s, %s)",
        (int(row['userId']), int(row['movieId']), 
         float(row['rating']), str(row['date']))
    )

conn.commit()
print("✅ Silver layer saved to MySQL!")

✅ Silver layer saved to MySQL!


In [10]:
cursor.execute("SELECT COUNT(*) FROM silver_movies")
print("Movies in Silver:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM silver_ratings")
print("Ratings in Silver:", cursor.fetchone()[0])

Movies in Silver: 9708
Ratings in Silver: 100836
